In [1]:
import pandas as pd
import numpy as np
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory


In [4]:
df = pd.read_excel('exerc8.xlsx', index_col='maquinas')

In [5]:

df

,custo_fixo,custo_variavel,capacidade
maquinas,,,
Alfa,3000,63,1500
Beta,2850,69,1800
Delta,2250,72,2250


In [13]:
model = pyo.ConcreteModel()

model.maquinas = pyo.Set(initialize=df.index.tolist())
model.custo_fixo = pyo.Param(model.maquinas, initialize=df['custo_fixo'].to_dict())
model.custo_variavel = pyo.Param(model.maquinas, initialize=df['custo_variavel'].to_dict())
model.capacidade = pyo.Param(model.maquinas, initialize=df['capacidade'].to_dict())

model.x = pyo.Var(model.maquinas, domain=pyo.NonNegativeIntegers)

In [14]:
#objetivo

def objetivo(model):
    return sum(model.x[m]*model.custo_variavel[m] + model.custo_fixo[m] for m in model.maquinas)
model.obj = pyo.Objective(rule=objetivo, sense=pyo.minimize)

In [15]:
#restrição de capacidade
def capacidade_regra(model, m):
    return model.x[m] <= model.capacidade[m]

model.capacidade_model = pyo.Constraint(model.maquinas, rule=capacidade_regra)

#restricao de demanda total

def demanda_total(model):
    return sum(model.x[m] for m in model.maquinas) >= 3900
model.demanda_total_model = pyo.Constraint(rule=demanda_total)

In [16]:
model.pprint()

1 Set Declarations
    maquinas : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'Alfa', 'Beta', 'Delta'}

3 Param Declarations
    capacidade : Size=3, Index=maquinas, Domain=Any, Default=None, Mutable=False
        Key   : Value
         Alfa :  1500
         Beta :  1800
        Delta :  2250
    custo_fixo : Size=3, Index=maquinas, Domain=Any, Default=None, Mutable=False
        Key   : Value
         Alfa :  3000
         Beta :  2850
        Delta :  2250
    custo_variavel : Size=3, Index=maquinas, Domain=Any, Default=None, Mutable=False
        Key   : Value
         Alfa :    63
         Beta :    69
        Delta :    72

1 Var Declarations
    x : Size=3, Index=maquinas
        Key   : Lower : Value : Upper : Fixed : Stale : Domain
         Alfa :     0 :  None :  None : False :  True : NonNegativeIntegers
         Beta :     0 :  None :  None : False :  True : NonNegativeIntegers
        Delta :  

In [17]:
res = SolverFactory('cplex')
opt = res.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\joaon\AppData\Local\Temp\tmp54y1vna5.cplex.log' open.
CPLEX> Problem 'C:\Users\joaon\AppData\Local\Temp\tmpm0tpk31h.pyomo.lp' read.
Read time = 0.02 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\joaon\AppData\Local\Temp\tmpm0tpk31h.pyomo.lp
Objective sense      : Minimize
Variables            :       4  [Fix: 1,  General Integer: 3]
Objective nonzeros   :       4
Linear constraints   :       4  [Less: 3,  Greater: 1]
  Nonzeros           :       6
  RHS nonzeros       :       4

Variables            : Min LB: 0.000000         Max UB: 1.000000       
Objective nonzeros   : M

In [25]:
for res in model.x:
    print(f'Maquina {res}: {model.x[res].value}')

print(f'Custo total: RS {model.obj():.2f}')

Maquina Alfa: 1500.0
Maquina Beta: 1800.0
Maquina Delta: 600.0
Custo total: RS 270000.00
